In [ ]:
# Cell 1: Setup and Libraries
!pip install easyocr

import cv2
import torch
import easyocr
import re
import difflib
import json
from collections import Counter
from google.colab import files

In [ ]:
# Cell 2: Image Upload and OCR Extraction
uploaded = files.upload()
image_path = next(iter(uploaded.keys()))

image = cv2.imread(image_path)
if image is None:
    raise ValueError("Could not read the uploaded image.")

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
scale = 2.0 if max(gray.shape[:2]) < 2000 else 1.0
if scale != 1.0:
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
gray = clahe.apply(gray)
ocr_image = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

gpu_available = torch.cuda.is_available()
reader = easyocr.Reader(['ar', 'en'], gpu=gpu_available)
results = reader.readtext(
    ocr_image, detail=1, paragraph=False, width_ths=1.2, height_ths=0.6, y_ths=0.5
)

items = []
for bbox, text, confidence in results:
    text = text.strip()
    if not text: continue
    xs = [p[0] for p in bbox]
    ys = [p[1] for p in bbox]
    items.append({
        "text": text, "confidence": confidence,
        "x": (min(xs) + max(xs)) / 2, "y": (min(ys) + max(ys)) / 2,
        "height": max(ys) - min(ys)
    })

items.sort(key=lambda x: x["y"])
lines = []
for item in items:
    placed = False
    for line in lines:
        avg_y = sum(x["y"] for x in line) / len(line)
        avg_height = sum(x["height"] for x in line) / len(line)
        if abs(item["y"] - avg_y) <= avg_height * 0.7:
            line.append(item)
            placed = True
            break
    if not placed:
        lines.append([item])

arabic_to_english = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize(text):
    text = text.translate(arabic_to_english)
    text = text.replace("٫", ".").replace("،", ",").replace("٬", ",")
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652\u0640]', '', text)
    return text

ocr_lines = []
for line in lines:
    line.sort(key=lambda x: x["x"], reverse=True)
    ocr_lines.append(normalize(" ".join(x["text"] for x in line)))

print("\n================ OCR LINES ================\n")
for i, line in enumerate(ocr_lines):
    print(f"{i} : {line}")

In [ ]:
# Cell 3: Fuzzy Matching and Multi-Strategy Number Extraction
def fuzzy_ratio(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()

def best_fuzzy_line(target_variants, threshold=0.55):
    best = (None, None, 0.0)
    for i, line in enumerate(ocr_lines):
        for kw in target_variants:
            klen = len(kw)
            for start in range(0, max(1, len(line) - klen + 3)):
                window = line[start:start + klen + 3]
                score = fuzzy_ratio(window, kw)
                if score > best[2]:
                    best = (i, kw, score)
    return best if best[2] >= threshold else (None, None, 0.0)

KWH_UNIT_VARIANTS = ["ك و س", "كوس", "ك.و.س", "كيلو وات", "kwh"]
CURRENCY_VARIANTS = ["ريال", "ر.س", "﷼", "sar"]

MAX_RAW_DIGITS = 7
AMOUNT_RANGE = (0.5, 100000)
CONSUMPTION_RANGE = (0.5, 100000)

def isolate_segment(text, start, max_len=40):
    seg = text[start:start + max_len]
    cut = re.search(r'[^\d\s.,:()\-]{2,}', seg)
    if cut:
        seg = seg[:cut.start()]
    return seg

def parse_numbers_from_segment(segment):
    if not segment: return []
    segment = normalize(segment)
    tokens = re.findall(r'[\d.,]+', segment)
    if not tokens: return []

    candidates = set()

    def add_float(s):
        s = s.replace(',', '.')
        if s.count('.') > 1:
            parts = s.split('.')
            s = "".join(parts[:-1]) + "." + parts[-1]
        if '.' in s:
            integer_part, decimal_part = s.split('.')
            if len(decimal_part) >= 3:
                s = integer_part + decimal_part

        s = s.strip('.,')
        if not s: return
        if len(s) > MAX_RAW_DIGITS + 3: return

        try:
            base_val = float(s)
            candidates.add(round(base_val, 2))
        except ValueError:
            return

        if '.' not in s and base_val > 99:
            if len(s) >= 4 and s[-3] == '0':
                fixed_s = s[:-3] + '.' + s[-2:]
                try: candidates.add(round(float(fixed_s), 2))
                except ValueError: pass
            elif len(s) >= 3 and s[-2] == '0':
                fixed_s = s[:-2] + '.' + s[-1:]
                try: candidates.add(round(float(fixed_s), 2))
                except ValueError: pass
            else:
                if base_val > 1000:
                    fixed_s = f"{s[:-2]}.{s[-2:]}"
                    try: candidates.add(round(float(fixed_s), 2))
                    except ValueError: pass

    clean_tokens = [t for t in tokens if re.search(r'\d', t)]
    if not clean_tokens: return []

    tokens_a = ['.' if t == '0' else t for t in clean_tokens]
    add_float("".join(reversed(tokens_a)))
    add_float("".join(reversed(clean_tokens)))
    add_float("".join(clean_tokens))
    tokens_d = [t for t in tokens if t.strip()]
    add_float("".join(reversed(tokens_d)))

    for t in tokens:
        if len(t) >= 3 and ('.' in t or ',' in t):
            add_float(t)

    return list(candidates)

def find_unit_anchored_number(unit_variants, plausible_range):
    candidates = []
    for line in ocr_lines:
        for uv in unit_variants:
            pos = line.lower().find(uv.lower())
            if pos == -1:
                for start in range(0, max(1, len(line) - len(uv) + 1)):
                    if fuzzy_ratio(line[start:start + len(uv)], uv) >= 0.8:
                        pos = start
                        break
            if pos == -1: continue

            before = isolate_segment(line[:pos][::-1], 0, 15)[::-1]
            after = isolate_segment(line, pos + len(uv), 15)
            for seg in (before, after):
                vals = parse_numbers_from_segment(seg)
                candidates.extend([v for v in vals if plausible_range[0] <= v <= plausible_range[1]])
    return candidates

In [ ]:
# Cell 4: Field-Specific Extraction and Double-Check Logic
def find_keyword_position(line, keyword_variants, threshold=0.72):
    for kw in keyword_variants:
        found = line.find(kw)
        if found != -1: return found + len(kw)
    best = (None, 0.0)
    for kw in keyword_variants:
        klen = len(kw)
        for start in range(0, max(1, len(line) - klen + 1)):
            score = fuzzy_ratio(line[start:start + klen], kw)
            if score > best[1]: best = (start + klen, score)
    return best[0] if best[1] >= threshold else None

def extract_value_after_keyword(keyword_variants, plausible_range):
    idx, _, _ = best_fuzzy_line(keyword_variants)
    if idx is None: return []
    line = ocr_lines[idx]
    pos = find_keyword_position(line, keyword_variants)
    if pos is None: return []

    segment = isolate_segment(line, pos, max_len=40)
    vals = parse_numbers_from_segment(segment)

    if not vals and idx + 1 < len(ocr_lines):
        next_line = ocr_lines[idx + 1]
        chunks = re.findall(r'(?:[\d]+[\s.,]*)+[\d]+', next_line)
        if chunks:
            vals = parse_numbers_from_segment(chunks[-1])

    return [v for v in vals if plausible_range[0] <= v <= plausible_range[1]]

def consensus(weighted_values, rel_tol=0.03):
    valid = [(v, w) for v, w in weighted_values if v is not None and v > 0]
    if not valid: return None

    scores = Counter()
    for v, w in valid:
        for other_v, other_w in valid:
            if v == other_v:
                scores[round(v, 2)] += w
            elif abs(v - other_v) <= max(v, other_v) * rel_tol:
                scores[round(v, 2)] += (w * 0.5)

    return scores.most_common(1)[0][0]

# Extract candidates exclusively for the amount due field
primary_amount_candidates = extract_value_after_keyword(["المبلغ المطلوب سداده", "المبلغ المطلوب سدادة"], AMOUNT_RANGE)

amount_candidates = []
for v in primary_amount_candidates:
    amount_candidates.append((v, 4.0)) # Very heavy weight for the target amount due field

for kws in [
    ["المبلغ شامل الضريبة", "المبلغ شامل الصربه", "المبلغ شامل الصريبه"],
    ["إجمالي الفاتورة", "إجمالى الفاتورة", "اجمالي الفاتوره"],
]:
    for v in extract_value_after_keyword(kws, AMOUNT_RANGE):
        amount_candidates.append((v, 1.0))

for v in find_unit_anchored_number(CURRENCY_VARIANTS, AMOUNT_RANGE):
    amount_candidates.append((v, 1.0))

consumption_candidates = []
for v in extract_value_after_keyword(["إجمالي الاستهلاك", "إجمالى الاستهلاك"], CONSUMPTION_RANGE):
    consumption_candidates.append((v, 3.0))

for v in extract_value_after_keyword(["كمية الاستهلاك", "كمبه الاستهلاك"], CONSUMPTION_RANGE):
    consumption_candidates.append((v, 1.0))

for v in find_unit_anchored_number(KWH_UNIT_VARIANTS, CONSUMPTION_RANGE):
    consumption_candidates.append((v, 1.0))

def calculate_expected_bill(consumption, meter_fee):
    if consumption <= 6000:
        val = consumption * 0.18
    else:
        val = (6000 * 0.18) + ((consumption - 6000) * 0.30)
    return (val + meter_fee) * 1.15

def strict_double_check(primary_amounts, amount_cands, cons_cands):
    best_match = None
    min_error = float('inf')

    primary_set = set(primary_amounts)
    c_set = set(c for c, w in cons_cands if c is not None and c > 0)

    # Step A: First search in matches that align with the explicit "Amount Due" field
    for a in primary_set:
        for c in c_set:
            for fee in [10.0, 15.0, 21.0, 30.0]:
                expected_a = calculate_expected_bill(c, fee)
                error = abs(a - expected_a) / max(a, 1)
                if error < min_error:
                    min_error = error
                    best_match = (a, c)

    if best_match and min_error <= 0.05: # Error margin below 5% is considered 100% verified match
        return best_match, min_error, "Verified via Primary Field & Tariff Match"

    # Step B: If no direct match is found, fallback to protected weighted consensus system
    return None, min_error, "Failed strict check, fallback to weighted consensus"

flat_amount_cands = list(set([v for v, w in amount_candidates]))
flat_cons_cands = list(set([v for v, w in consumption_candidates]))

double_check_result, err_margin, check_status = strict_double_check(primary_amount_candidates, amount_candidates, consumption_candidates)

if double_check_result:
    amount_due, consumption_value = double_check_result
    print(f"\n✅ Strict double check passed ({check_status})! Mathematical error rate: {err_margin:.2%}")
else:
    amount_due = consensus(amount_candidates)
    consumption_value = consensus(consumption_candidates)
    print(f"\n⚠️ Notice: Fell back to weighted consensus system.")

In [ ]:
# Cell 5: Date Parsing and Final JSON Output
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]

def parse_date_segment(segment):
    tokens = re.findall(r'\d+', normalize(segment))
    if not tokens: return None
    str_reversed = "".join(reversed(tokens))
    str_normal = "".join(tokens)
    for candidate in (str_reversed, str_normal):
        year_match = re.search(r'(20[1-3][0-9])', candidate)
        if not year_match: continue
        y = year_match.group(1)
        rem = candidate.replace(y, "", 1)
        parts = [int(p) for p in re.split(r'0+', rem) if p]
        p1 = p2 = None
        if len(parts) >= 2:
            p1, p2 = parts[0], parts[1]
        elif len(parts) == 1:
            s = str(parts[0])
            if len(s) == 4: p1, p2 = int(s[:2]), int(s[2:])
            elif len(s) == 3: p1, p2 = (int(s[:2]), int(s[2:])) if int(s[:2]) <= 31 else (int(s[:1]), int(s[1:]))
        if p1 is None or p2 is None: continue
        m = None
        if p1 > 12 >= p2: m = p2
        elif p2 > 12 >= p1: m = p1
        elif p1 <= 12 and p2 <= 12: m = p1 if candidate.startswith(y) else p2
        if m and 1 <= m <= 12: return f"{MONTHS[m - 1]} {y}"
    return None

invoice_month = None
DATE_KEYWORDS = [
    ["تاريخ الفاتورة", "تاريخ الفاتوره"], ["بداية الاستحقاق"],
    ["نهاية الفترة", "نهاية الفترذ"], ["بداية الفترة"],
]
for kws in DATE_KEYWORDS:
    if invoice_month: break
    idx, _, _ = best_fuzzy_line(kws)
    if idx is None: continue
    line = ocr_lines[idx]
    pos = find_keyword_position(line, kws)
    if pos is None: continue
    segment = isolate_segment(line, pos, max_len=25)
    invoice_month = parse_date_segment(segment)

invoice_data = {
    "month": invoice_month,
    "amount_due": amount_due,
    "amount_candidates": flat_amount_cands,
    "consumption": consumption_value,
    "consumption_candidates": flat_cons_cands,
}

print("\n==========================================")
print("            INVOICE RESULT")
print("==========================================")
print("Month:", invoice_data["month"])
print("Amount Due:", invoice_data["amount_due"], "SAR")
print("Consumption:", invoice_data["consumption"], "kWh")
print("\n================ JSON ================\n")
print(json.dumps(invoice_data, ensure_ascii=False, indent=4))